# 整合特征 GFF：全基因组功能区域标注（IRGSP-1.0）· 链特异性版本

本 notebook 把水稻参考基因组（IRGSP-1.0，chr01–12，共 ~373 Mb）的**每一个碱基**归类到 7 类**互斥**的功能特征，并输出一个整合的 GFF3 文件：

- **Intergenic**：所有已注释区域之外的背景区
- **CDS**：编码区
- **Intron**：由外显子 gaps **推导**而来
- **5_UTR** / **3_UTR**：非翻译区
- **ncRNA**：rRNA / tRNA
- **Repeat**：转座子 / 串联重复等重复单元

**用途**：为跨模型功能 embedding 对比（muxi-1B、NTv3、PlantCAD2、GENERator、OryzaG3、Botanic/agront 等）提供**统一、链特异性、全基因组覆盖**的区域标签——下游可按 `feature + strand` 取序列（负链反互补）后做功能潜空间分析。

**链特异性设计**（v2 变更）：
1. 正链、负链的标注区间**各带自己的 strand**（`+` / `-`）进入合并与扫描线；
2. **正链、负链分别跑扫描线**，链内重叠处取优先级最高特征（CDS > 5_UTR > 3_UTR > Intron > ncRNA > Repeat）；
3. 同一段 DNA 可同时出现在 `+` 与 `-` 两行（反义转录 / 反义链基因）——这是保留链信息后的**必然结果**，不是 bug；
4. **Intergenic**：仅当两条链都无任何标注时才输出（strand 列 = `.`），每段只输出一次；
5. 不变量：每条链内每碱基**至多一个**标签；Intergenic 与所有标注**不重叠**；全部区域并集**无缝隙覆盖**全基因组。

## 数据源

| 特征 | 来源文件 | 原始 feature |
|---|---|---|
| CDS | `IRGSP-1.0_representative/transcripts.gff` | `CDS` |
| 5_UTR | 同上 | `five_prime_UTR` |
| 3_UTR | 同上 | `three_prime_UTR` |
| Intron | `IRGSP-1.0_representative/transcripts_exon.gff` | 由 `exon` gaps 推导（继承转录本链） |
| ncRNA | `irgsp1_rRNA_tRNA.gff.gz` | `rRNA`, `tRNA` |
| Repeat | `irgsp1_repeat_unit.gff.gz` | `repeat_unit`（~28.4 万条） |
| Intergenic | 正、负链**都**未标注的区域 | 两套扫描线结果取补集交集 |

> 说明：Repeat 文件含克隆/细胞器等非核染色体，仅取 chr01–12 核基因组。
> 各源 GFF 的 strand 列都已带 `+`/`-`（实测 ± 链各约一半），本流程负责把它们保留到输出。

In [1]:
import gzip
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# ---------------- 路径配置（可按需修改） ----------------
BASE = Path("/mnt/rice/default/Workspace/yangdong/ai4research/DATA/rice")
OUT  = Path("/mnt/rice/default/Workspace/yangdong/ai4research/new_project/evaluate/func_embedding/output")

GFF_TRANSCRIPTS = BASE / "IRGSP-1.0_representative" / "transcripts.gff"      # CDS / UTR
GFF_EXONS       = BASE / "IRGSP-1.0_representative" / "transcripts_exon.gff" # exon / mRNA -> Intron
GFF_NCRNA       = BASE / "irgsp1_rRNA_tRNA.gff.gz"                           # rRNA / tRNA
GFF_REPEAT      = BASE / "irgsp1_repeat_unit.gff.gz"                         # repeat_unit
GENOME_FASTA    = BASE / "IRGSP-1.0_genome.fasta"                            # 染色体长度

In [3]:
# ---------------- 常量 ----------------
CHROMS = [f"chr{i:02d}" for i in range(1, 13)]

# 特征类型与优先级（下标越小优先级越高；重叠区段按此顺序分配标签）
FEATURE_ORDER = ["CDS", "5-UTR", "3-UTR", "Intron", "ncRNA", "Repeat", "Intergenic"]
PRIORITY      = {t: i for i, t in enumerate(FEATURE_ORDER[:-1])}  # Intergenic 由补集得到

# 链 token：内部扫描用 ±1（每链一套标注），写 GFF3 时映射回字符串
STRAND_TOKENS = {"+": 1, "-": -1}
TOKEN_STR     = {1: "+", -1: "-"}

# 染色体长度：优先从 FASTA/FAI 读取，失败则用官方 IRGSP-1.0 长度兜底
try:
    from pyfaidx import Fasta
    CHROM_SIZE = {rec.name: len(rec) for rec in Fasta(str(GENOME_FASTA))}
except Exception:
    CHROM_SIZE = {"chr01": 43270923, "chr02": 35937250, "chr03": 36413819,
                  "chr04": 35502694, "chr05": 29958434, "chr06": 31248787,
                  "chr07": 29697621, "chr08": 28443022, "chr09": 23012720,
                  "chr10": 23207287, "chr11": 29021106, "chr12": 27531856}
print({c: CHROM_SIZE[c] for c in CHROMS})

{'chr01': 43270923, 'chr02': 35937250, 'chr03': 36413819, 'chr04': 35502694, 'chr05': 29958434, 'chr06': 31248787, 'chr07': 29697621, 'chr08': 28443022, 'chr09': 23012720, 'chr10': 23207287, 'chr11': 29021106, 'chr12': 27531856}


## Step 1 · 读取 GFF 的辅助函数

统一把 GFF 读成 9 列 DataFrame（`start`/`end` 转 int64），自动支持 `.gz`。

In [4]:
def read_gff(path: Path, features=None, chroms=None) -> pd.DataFrame:
    """读取 GFF（自动支持 .gz），返回 9 列 DataFrame（start/end 已 int 化，1-based 闭区间）。"""
    open_func = gzip.open if str(path).endswith(".gz") else open
    rows = []
    with open_func(path, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            p = line.rstrip("\n").split("\t")
            if len(p) < 9:
                continue
            if chroms is not None and p[0] not in chroms:
                continue
            if features is not None and p[2] not in features:
                continue
            rows.append(p)
    df = pd.DataFrame(rows, columns=["chr", "source", "feature", "start", "end",
                                     "score", "strand", "frame", "attributes"])
    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)
    return df

## Step 2 · 基因结构注释：CDS / 5′UTR / 3′UTR

`transcripts.gff` 已直接给出 CDS 与 UTR 区间，无需推导；按染色体压入 `interval_map`（记录 `[start, end, strand_token, 优先级]`），**每条链独立记录**。

In [5]:
interval_map = {c: [] for c in CHROMS}

ts = read_gff(GFF_TRANSCRIPTS,
              features={"CDS", "five_prime_UTR", "three_prime_UTR"},
              chroms=set(CHROMS))
print(ts.feature.value_counts().to_string())

src = {"CDS": "CDS", "five_prime_UTR": "5-UTR", "three_prime_UTR": "3-UTR"}
for gff_feat, out_feat in src.items():
    rank = PRIORITY[out_feat]
    for r in ts[ts.feature == gff_feat].itertuples():
        # 区间元组: (start, end, strand_token, priority_rank)
        interval_map[r.chr].append((r.start, r.end, STRAND_TOKENS[r.strand], rank))

feature
CDS                156214
three_prime_UTR     45840
five_prime_UTR      39870


In [6]:
def compute_introns(exons_df: pd.DataFrame) -> pd.DataFrame:
    """由转录本外显子 gaps 推导内含子区间（内含子继承转录本的链）。

    transcripts_exon.gff 中 mRNA 行提供 ID=... 与跨度、strand，exon 行提供 Parent=...。
    对每个转录本：按 start 排序其 exon，相邻 exon 之间的空隙即 Intron。
    """
    mrna  = exons_df[exons_df.feature == "mRNA"]
    exons = exons_df[exons_df.feature == "exon"]

    info_of = {}
    for r in mrna.itertuples():
        mid = r.attributes.split("ID=", 1)[1].split(";", 1)[0]
        info_of[mid] = (r.chr, r.strand)

    parent = exons["attributes"].str.extract(r"Parent=([^;]+)", expand=False).fillna("")
    introns = []
    for tid, grp in exons.assign(_p=parent.values).groupby("_p"):
        if tid not in info_of:
            continue
        chrom, strand = info_of[tid]
        ex = grp[["start", "end"]].to_numpy(dtype=np.int64)
        ex = ex[ex[:, 0].argsort(kind="stable")]
        prev_end = None
        for s0, e0 in ex:
            s0, e0 = int(s0), int(e0)
            if prev_end is not None and s0 > prev_end + 1:
                introns.append((chrom, prev_end + 1, s0 - 1, strand))
            prev_end = e0 if prev_end is None else max(prev_end, e0)
    return pd.DataFrame(introns, columns=["chr", "start", "end", "strand"])


t0 = time.time()
exons_df  = read_gff(GFF_EXONS, features={"mRNA", "exon"}, chroms=set(CHROMS))
intron_df = compute_introns(exons_df)
print(f"Intron 推导完成: {len(intron_df):,} 条, 用时 {time.time()-t0:.1f}s")

for r in intron_df.itertuples():
    interval_map[r.chr].append((r.start, r.end, STRAND_TOKENS[r.strand], PRIORITY["Intron"]))

Intron 推导完成: 140,707 条, 用时 11.1s


## Step 3 · ncRNA 与 Repeat

- **ncRNA**：`irgsp1_rRNA_tRNA.gff.gz` 中的 `rRNA` / `tRNA` 亲本区间（不含 tRNA_exon 子特征）。
- **Repeat**：`irgsp1_repeat_unit.gff.gz` 中的 `repeat_unit`（chr01–12 共 ~28.4 万条）。

In [7]:
t0 = time.time()
ncrna_df = read_gff(GFF_NCRNA, features={"rRNA", "tRNA"}, chroms=set(CHROMS))
for r in ncrna_df.itertuples():
    interval_map[r.chr].append((r.start, r.end, STRAND_TOKENS[r.strand], PRIORITY["ncRNA"]))
print(f"ncRNA: {len(ncrna_df):,}  ({time.time()-t0:.1f}s)")

t0 = time.time()
rep_df = read_gff(GFF_REPEAT, features={"repeat_unit"}, chroms=set(CHROMS))
for r in rep_df.itertuples():
    interval_map[r.chr].append((r.start, r.end, STRAND_TOKENS[r.strand], PRIORITY["Repeat"]))
print(f"Repeat: {len(rep_df):,}  ({time.time()-t0:.1f}s)")

ncRNA: 955  (0.0s)
Repeat: 283,848  (1.4s)


## Step 4 · 扫描线分区：每条链唯一标签

算法（**每条链各跑一遍**）：
1. 亲和反义重叠：链上相同位置**不同链**的区间**不合并**、不互相覆盖——`+` 链与 `-` 链分别归并；
2. 每种特征区间先**合并**为不相交 block（间距 ≤1 bp 视为相连）；
3. 所有 block 端点按坐标排序，扫描线维护"当前活跃特征"，重叠处取优先级最高者；
4. 某位置两条链都无标注 → 各自扫描线返回 `None` → 该位置归 **Intergenic**（只输出一次）。

复杂度 O(N log N)，全基因组（~62 万条各类区间 + 合并后 ~46 万 block）约十几秒。

In [8]:
def merge_intervals(starts: np.ndarray, ends: np.ndarray) -> list[tuple[int, int]]:
    """经典区间合并（1-based 闭区间，间隔 <=1 视为相连）。"""
    if len(starts) == 0:
        return []
    order = np.argsort(starts, kind="stable")
    s, e = starts[order], ends[order]
    merged = []
    cs, ce = int(s[0]), int(e[0])
    for i in range(1, len(s)):
        ns, ne = int(s[i]), int(e[i])
        if ns <= ce + 1:
            ce = max(ce, ne)
        else:
            merged.append((cs, ce))
            cs, ce = ns, ne
    merged.append((cs, ce))
    return merged


def label_partition(chrom_size: int, intervals):
    """扫描线：给 [1, chrom_size] 每个位置分配"优先级最高"的活跃特征标签。

    intervals: [(start, end, priority_rank)]；返回 [(start, end, rank_or_None)]。
    rank 为 None 表示无任何特征覆盖 -> Intergenic。
    """
    events = []
    for s, e, rank in intervals:
        s, e = int(s), int(e)
        if e < s:
            continue
        s = max(s, 1); e = min(e, chrom_size)
        if e < s:
            continue
        events.append((s, +1, rank))
        events.append((e + 1, -1, rank))
    events.sort(key=lambda x: (x[0], -x[1], x[2]))

    active  = [0] * len(PRIORITY)
    regions, prev_pos = [], 1
    top = None

    def current_top():
        for r in range(len(active)):
            if active[r] > 0:
                return r
        return None

    i, n = 0, len(events)
    while i < n:
        pos = events[i][0]
        if pos > prev_pos:
            regions.append((prev_pos, pos - 1, top))
        j = i
        while j < n and events[j][0] == pos:      # 同一坐标的事件一起处理
            _, delta, rank = events[j]
            active[rank] += delta
            j += 1
        top = current_top()
        prev_pos = pos
        i = j
    if prev_pos <= chrom_size:
        regions.append((prev_pos, chrom_size, top))
    return regions

In [9]:
t0 = time.time()
region_rows = []
counter = {}

for chrom in CHROMS:
    intv = interval_map[chrom]
    annotated_blocks = []   # (start, end, strand_token, rank)：链内去重叠后的标注

    # 1) 每条链独立合并 + 扫描线（重叠处取优先级最高特征；不同链互不影响）
    for strand_tok in (1, -1):
        s_intv = [iv for iv in intv if iv[2] == strand_tok]
        merged = []
        for rank in range(len(PRIORITY)):
            parts = [iv for iv in s_intv if iv[3] == rank]
            if not parts:
                continue
            ss = np.array([p[0] for p in parts], dtype=np.int64)
            ee = np.array([p[1] for p in parts], dtype=np.int64)
            for s0, e0 in merge_intervals(ss, ee):
                merged.append((s0, e0, rank))
        for s0, e0, rank in label_partition(CHROM_SIZE[chrom], merged):
            if rank is not None:
                annotated_blocks.append((s0, e0, strand_tok, rank))
        # rank 为 None 的区间：本链无标注，是否 Intergenic 由两链并集补集决定

    # 2) 写出链特异性标注行（strand = +/-）
    for s0, e0, strand_tok, rank in annotated_blocks:
        ftype = FEATURE_ORDER[rank]
        counter[ftype] = counter.get(ftype, 0) + 1
        region_rows.append((chrom, ftype, s0, e0, TOKEN_STR[strand_tok],
                            f"ID={ftype}_{counter[ftype]}"))

    # 3) Intergenic = 两条链所有标注（跨链合并）的补集；strand="."，每段只输出一次
    if annotated_blocks:
        ss = np.array([b[0] for b in annotated_blocks], dtype=np.int64)
        ee = np.array([b[1] for b in annotated_blocks], dtype=np.int64)
        union = merge_intervals(ss, ee)
        prev_end = 0
        for s0, e0 in union:
            if s0 > prev_end + 1:
                ig = "Intergenic"
                counter[ig] = counter.get(ig, 0) + 1
                region_rows.append((chrom, ig, prev_end + 1, s0 - 1, ".",
                                    f"ID={ig}_{counter[ig]}"))
            prev_end = max(prev_end, e0)
        if prev_end < CHROM_SIZE[chrom]:
            ig = "Intergenic"
            counter[ig] = counter.get(ig, 0) + 1
            region_rows.append((chrom, ig, prev_end + 1, CHROM_SIZE[chrom], ".",
                                f"ID={ig}_{counter[ig]}"))
    else:
        ig = "Intergenic"
        counter[ig] = counter.get(ig, 0) + 1
        region_rows.append((chrom, ig, 1, CHROM_SIZE[chrom], ".",
                            f"ID={ig}_{counter[ig]}"))

regions_df = pd.DataFrame(region_rows,
                          columns=["chr", "feature", "start", "end", "strand", "id"])
regions_df["bp"] = regions_df.end - regions_df.start + 1
print(f"分区完成: {len(regions_df):,} 条区域, 用时 {time.time()-t0:.1f}s")
print(f"strand 分布: {regions_df.strand.value_counts().to_dict()}")
regions_df.head()

分区完成: 884,297 条区域, 用时 3.8s
strand 分布: {'+': 323188, '-': 314640, '.': 246469}


,chr,feature,start,end,strand,id,bp
0,chr01,Repeat,2292,2472,+,ID=Repeat_1,181
1,chr01,Repeat,2533,2613,+,ID=Repeat_2,81
2,chr01,Repeat,2772,2928,+,ID=Repeat_3,157
3,chr01,5-UTR,2983,3268,+,ID=5-UTR_1,286
4,chr01,Intron,3269,3353,+,ID=Intron_1,85


In [10]:
# ---- 正确性校验（链特异性版） ----
genome_bp = sum(CHROM_SIZE[c] for c in CHROMS)

# 1) 无缝隙：每条染色体上所有区域按"坐标并集"恰好覆盖整条染色体
for c in CHROMS:
    sub = regions_df[regions_df.chr == c]
    ints = sorted(zip(sub.start.tolist(), sub.end.tolist()))
    merged_bp = 0
    cur_s, cur_e = None, None
    for s0, e0 in ints:
        if cur_e is None:
            cur_s, cur_e = s0, e0
        elif s0 <= cur_e + 1:
            cur_e = max(cur_e, e0)
        else:
            merged_bp += cur_e - cur_s + 1
            cur_s, cur_e = s0, e0
    merged_bp += cur_e - cur_s + 1
    assert merged_bp == CHROM_SIZE[c], f"{c}: 覆盖 {merged_bp} != {CHROM_SIZE[c]}"

# 2) 链内唯一：同一链上同一碱基至多一个标注（Intergenic 无链语义，跳过）
for strand in ("+", "-"):
    sub = regions_df[regions_df.strand == strand].sort_values(["chr", "start"])
    prev_chr, prev_end = None, None
    for r in sub.itertuples():
        if prev_chr == r.chr and r.start <= prev_end:
            raise AssertionError(f"{strand} 链标注重叠 @ {r.chr}:{r.start}-{r.end}")
        prev_chr, prev_end = r.chr, r.end

# 3) Intergenic 恰好为两链标注并集的补集（无重叠、无缺失）
for c in CHROMS:
    sub = regions_df[regions_df.chr == c]
    ann = sub[sub.strand != "."][["start", "end"]].to_numpy(dtype=np.int64)
    ig  = sub[sub.strand == "."][["start", "end"]].to_numpy(dtype=np.int64)
    gaps = []
    if len(ann):
        merged_ann = merge_intervals(ann[:, 0], ann[:, 1])
        pe = 0
        for s0, e0 in merged_ann:
            if s0 > pe + 1:
                gaps.append((pe + 1, s0 - 1))
            pe = max(pe, e0)
        if pe < CHROM_SIZE[c]:
            gaps.append((pe + 1, CHROM_SIZE[c]))
    else:
        gaps = [(1, CHROM_SIZE[c])]
    assert sorted(gaps) == sorted((int(a), int(b)) for a, b in ig), \
        f"{c}: Intergenic 与标注重叠或缺段"

print(f"校验通过：全基因组 {genome_bp:,} bp 无缝隙/无重叠；"
      f"+/- 链各自唯一标签；Intergenic 为两链均未标注的区域。\n")

# ---- 反义信号统计（仅供参考）：+/- 链标注重叠的碱基数 ----
antisense_bp = 0
for c in CHROMS:
    sub   = regions_df[regions_df.chr == c]
    plus  = sub[sub.strand == "+"][["start", "end"]].to_numpy(dtype=np.int64)
    minus = sub[sub.strand == "-"][["start", "end"]].to_numpy(dtype=np.int64)
    if len(plus) == 0 or len(minus) == 0:
        continue
    mm = merge_intervals(minus[:, 0], minus[:, 1])
    for s0, e0 in merge_intervals(plus[:, 0], plus[:, 1]):
        for ms0, me0 in mm:
            if me0 < s0 or ms0 > e0:
                continue
            antisense_bp += min(e0, me0) - max(s0, ms0) + 1

print(f"反义重叠（+ 与 - 链标注重叠）: {antisense_bp:,} bp "
      f"({antisense_bp / genome_bp:.2%} of genome；每链独立标注，故意保留)")

校验通过：全基因组 373,245,519 bp 无缝隙/无重叠；+/- 链各自唯一标签；Intergenic 为两链均未标注的区域。

反义重叠（+ 与 - 链标注重叠）: 11,770,172 bp (3.15% of genome；每链独立标注，故意保留)


## Step 5 · 写出整合特征 GFF3

按 `chr, start` 排序后写为标准 GFF3（9 列，1-based 闭区间）：

| 列 | 值 |
|---|---|
| seqid | chr01–chr12 |
| source | `integrated_features` |
| feature | CDS / 5_UTR / 3_UTR / Intron / ncRNA / Repeat / Intergenic |
| strand | `+` / `-`（基因结构类）；`.`（Intergenic） |
| score / frame | `.` |
| attributes | `ID={Feature}_{全局序号}` |

> 下游注意：`strand == "-"` 的区域取序列时需要**反互补**（已与 `ogr_extract/genome.py fetch_seq` 兼容）。

写文件时特征名默认把连字符 `-` 转为下划线（`5-UTR` → `5_UTR`），与现有 `configs/models.yaml` 的 feature 列表一致。

In [11]:
OUT.mkdir(parents=True, exist_ok=True)
gff_path = OUT / "integrated_features.gff"

FEAT_TO_GFF = {t: t.replace("-", "_") for t in FEATURE_ORDER}   # 5-UTR -> 5_UTR

regions_df = regions_df.sort_values(["chr", "start"]).reset_index(drop=True)
with open(gff_path, "w") as fh:
    fh.write("##gff-version 3\n")
    fh.write("##source: integrated from IRGSP-1.0 annotation (CDS/UTR/Intron/ncRNA/Repeat), strand-aware\n")
    fh.write("##priority: CDS > 5_UTR > 3_UTR > Intron > ncRNA > Repeat > Intergenic\n")
    fh.write("##note: Intergenic has strand='.'; other features strand='+'/'-' per strand\n")
    for r in regions_df.itertuples():
        ftype = FEAT_TO_GFF[r.feature]
        fh.write(f"{r.chr}\tintegrated_features\t{ftype}\t{r.start}\t{r.end}"
                 f"\t.\t{r.strand}\t.\tID={ftype}_{r.Index + 1}\n")
print(f"写出 -> {gff_path}  ({len(regions_df):,} 行)")

写出 -> /mnt/rice/default/Workspace/yangdong/ai4research/new_project/evaluate/func_embedding/output/integrated_features.gff  (884,297 行)


In [12]:
# ---- 汇总统计（按 feature × strand，Intergenic 无链语义） ----
summary = (regions_df.groupby(["feature", "strand"], as_index=False)
           .agg(n_regions=("bp", "size"), bp=("bp", "sum")))
summary["frac_of_genome"] = summary.bp / genome_bp
summary = summary.sort_values(
    "feature",
    key=lambda s: s.map({t: i for i, t in enumerate(FEATURE_ORDER)}),
).sort_values(["feature", "strand"])
summary.to_csv(OUT / "integrated_features_summary.tsv", sep="\t", index=False)

with pd.option_context("display.float_format", "{:.4f}".format):
    print(summary.to_string(index=False))

# 链平衡概览（+ 与 - 的碱基比例接近 1:1，说明链信息完整保留）
if "strand" in summary:
    print("\n--- 链平衡（标注 bp） ---")
    print(summary[summary.strand != "."].pivot_table(
        index="feature", columns="strand", values="bp", aggfunc="sum",
        fill_value=0).assign(ratio=lambda d: d["+"] / d["-"]))

   feature strand  n_regions        bp  frac_of_genome
     3-UTR      +      22738   6609730          0.0177
     3-UTR      -      22389   6537971          0.0175
     5-UTR      +      20251   3653384          0.0098
     5-UTR      -      19661   3585230          0.0096
       CDS      +      79271  18706952          0.0501
       CDS      -      78222  18228268          0.0488
Intergenic      .     246469 123484239          0.3308
    Intron      +      73707  26123906          0.0700
    Intron      -      72361  26210532          0.0702
    Repeat      +     126505  76105080          0.2039
    Repeat      -     121580  75633877          0.2026
     ncRNA      +        716     96002          0.0003
     ncRNA      -        427     40520          0.0001

--- 链平衡（标注 bp） ---
strand          +         -     ratio
feature                              
3-UTR     6609730   6537971  1.010976
5-UTR     3653384   3585230  1.019010
CDS      18706952  18228268  1.026261
Intron   26123906  2

## Step 6 · 可视化

从汇总可见：**Repeat ~39%**、**Intergenic ~33%**、**Intron ~13%**、**CDS ~10%**——与水稻基因组组成一致（转座子/重复元件的广泛扩张是水稻基因组最显著的特征之一）。

链特异性版新增说明：$+/-$ 链各约一半的标注 bp（比值 ≈ 1.0）证明链信息完整保留；少量 $+/-$ 链标注重叠区域（反义转录/反义基因）会被重复输出，这是特性而非 bug。

In [13]:
palette = {"CDS": "#d62728", "5-UTR": "#ff7f0e", "3-UTR": "#fdbe17",
           "Intron": "#2ca02c", "ncRNA": "#17becf", "Repeat": "#8c564b",
           "Intergenic": "#b0b0b0"}

## 输出文件与下游用途

| 文件 | 内容 |
|---|---|
| `output/integrated_features.gff` | 全基因组链特异性特征分区 GFF3（7 类，~89 万条+标注行）；Intergenic strand=`.`，其余为 `+`/`-` |
| `output/integrated_features_summary.tsv` | 每类 × 每链特征：区域数、总 bp、占基因组比例 |
| `output/feature_composition.png` | 染色体组成（左）、全基因组占比（中）、链平衡（右） |

**下游使用**：对接 `extract_gene_embeddings.py` 同款管线（pyfaidx 取序列 → 单碱基 token → mask-aware 池化），可扩展为"按特征分区取 embedding"：读本 GFF 的任一特征区间 → 从 `IRGSP-1.0_genome.fasta` 取对应序列（负链自动反互补）→ 各模型 embedding → 按 CDS / Intron / Intergenic / Repeat 等分组做功能潜空间分析。

> 注意：`+`/`-` 链反义重叠区会各输出一行；Analysis 时按 `feature-strand` 分组即可。Intergenic 无链语义，取序列用正链即可。